In [0]:
%run ../00-common/01.Environment-config

In [0]:
control_table = f"{catalog_name}.{control_schema}.batch_control"

In [0]:
from pyspark.sql import functions as F

In [0]:
landing_batches = sorted({
    file.name.rstrip("/")
    for file in dbutils.fs.ls(landing_folder_path)
    if file.isDir()
})

In [0]:
if spark.catalog.tableExists(control_table):
    tracked_batches = [
        row.batch_id
        for row in (
            spark.table(control_table)
                 .filter(F.col("status").isin("in_progress","completed"))
                 .select("batch_id")
                 .distinct()
                 .collect()
        )
    ]
else:
    tracked_batches = []

In [0]:
new_batches = sorted(list(set(landing_batches) - set(tracked_batches)))
next_batch = new_batches[0] if new_batches else None

In [0]:
print(f'landing batches : {landing_batches}')
print(f'tracked_batches : {tracked_batches}')
print(f'next batch to progress : {next_batch}')

In [0]:
if next_batch is None:
    dbutils.jobs.taskValues.set(key='p_batch_id', value='')
    dbutils.jobs.taskValues.set(key='has_batch', value='false')
else:
    dbutils.jobs.taskValues.set(key='p_batch_id',value=next_batch)
    dbutils.jobs.taskValues.set(key='has_batch',value='true')